<div style="display: flex; justify-content: space-between;">
<div style="text-align: left; display: inline-block;" align="left"><b>Tepper School of Business </b></div>
<div style="text-align: right; display: inline-block;" align="right"><i>Copyright Dennis Epple</i></div>
</div>
<hr>
<div style="display: flex; justify-content: space-between;">
<div style="text-align: left" align="left">Statistical Decision Making (45-752)</div>
</div>

In [ ]:
# This command installs tprstats from its GitHub repo. You only need to run this command once, when start the notebook.
#!pip install git+https://github.com/dnepple/tprstats-python@colab

# stress test to see if this is a good solution to problem of installing tprstats
try:
    import tprstats
except ImportError as e:
  !pip install git+https://github.com/dnepple/tprstats-python@colab
  import tprstats

In [ ]:
# import the google drive package
from google.colab import drive
# mount google drive
drive.mount('/content/drive')
# set the working directory to the course folder
%cd '/content/drive/MyDrive/SDM'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/SDM


In [ ]:
import pandas
import numpy
import matplotlib.pyplot as plt

# Notebook 4

### Scaling Coefficients

In [ ]:
# import dataset Diamonds_211
Diamonds_211 = pandas.read_excel("data/Diamonds_211.xlsx")
# Import Water_Conservation_Data
Water_Conservation_Data=pandas.read_excel("data/Water_Conservation_Data.xlsx")
# Import Parcel Delivery Data Set
Parcel_Delivery = pandas.read_excel("data/Parcel_Delivery.xlsx")

In [ ]:
#  Estimate Diamond regression
DiamReg=tprstats.model("cs", "Price~Carat+Clarity",Diamonds_211)

In [ ]:
# view the regression
DiamReg.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                  Price   R-squared:                       0.950
Model:                            OLS   Adj. R-squared:                  0.950
No. Observations:                 211   F-statistic:                     773.8
Covariance Type:                  HC1   Prob (F-statistic):           4.54e-97
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   -1.16e+04    759.549    -15.278      0.000   -1.31e+04   -1.01e+04
Carat       1.193e+04    314.094     37.973      0.000    1.13e+04    1.25e+04
Clarity     1308.8177    124.664     10.499      0.000    1063.051    1554.584
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity robust (HC1)
"""

In [ ]:
# view summary statistics for the data
Diamonds_211.describe()

,Price,Carat,Clarity,Color,Cut,Salesperson
count,211.000000,211.000000,211.000000,211.0,211.0,211.000000
mean,8769.364929,1.239573,4.270142,11.0,2.0,5.208531
std,5744.400127,0.511450,1.206336,0.0,0.0,2.469649
min,2056.000000,0.510000,2.000000,11.0,2.0,1.000000
25%,4037.500000,0.800000,3.000000,11.0,2.0,3.000000
50%,6914.000000,1.110000,4.000000,11.0,2.0,5.000000
75%,11750.500000,1.585000,5.000000,11.0,2.0,7.000000
max,28546.000000,2.470000,8.000000,11.0,2.0,9.000000


In [ ]:
# Calculate and interpret scaled coefficients
DiamReg.scaled_coefficients()

,coefs,std_coefs,elasticities
Carat,11927.145390,1.061928,1.6859
Clarity,1308.817663,0.274854,0.6373


## Regression with Indicator Variables


In [ ]:
# Select the subset of observations in treatment group 3 and the control group
WatDat34=Water_Conservation_Data.query('GROUP==3|GROUP==4')

In [ ]:
# Run regression to estimate effect of treatment 3
WatReg34=tprstats.model("cs", "SUMMER_07~TREAT3",WatDat34)

In [ ]:
# view the regression result
WatReg34.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:              SUMMER_07   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                  0.000
No. Observations:               83319   F-statistic:                     36.54
Covariance Type:                  HC1   Prob (F-statistic):           1.50e-09
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     36.4725      0.109    333.730      0.000      36.258      36.687
TREAT3        -1.6145      0.267     -6.045      0.000      -2.138      -1.091
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity robust (HC1)
"""

In [ ]:
# Estimate all treatment effects at once and view the results.
WatRegAll_07=tprstats.model("cs", "SUMMER_07~TREAT1+TREAT2+TREAT3", Water_Conservation_Data)
WatRegAll_07.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:              SUMMER_07   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                  0.000
No. Observations:              106669   F-statistic:                     14.99
Covariance Type:                  HC1   Prob (F-statistic):           9.43e-10
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     36.4725      0.109    333.728      0.000      36.258      36.687
TREAT1        -0.1197      0.302     -0.396      0.692      -0.712       0.472
TREAT2        -1.0451      0.282     -3.701      0.000      -1.599      -0.492
TREAT3        -1.6145      0.267     -6.045      0.000      -2.138      -1.091
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity robust (HC1)
"""

The command below tests the null hypothesis that all three treatment effects are the same.

In [ ]:
# Test null hypothesis that all populationtreatment effects are the same.
WatRegAll_07.wald_test(("TREAT1=TREAT2, TREAT2=TREAT3"))

p-value:  0.0003


In [ ]:
# Estimate treatment effects for Summer 2009
WatRegAll_09=tprstats.model("cs", "SUMMER_09~TREAT1+TREAT2+TREAT3", Water_Conservation_Data)

In [ ]:
# view regression results
WatRegAll_09.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:              SUMMER_09   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                  0.000
No. Observations:              106669   F-statistic:                     1.672
Covariance Type:                  HC1   Prob (F-statistic):              0.171
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     27.4558      0.078    350.668      0.000      27.302      27.609
TREAT1         0.3005      0.210      1.433      0.152      -0.110       0.711
TREAT2        -0.0865      0.210     -0.412      0.680      -0.498       0.325
TREAT3        -0.2838      0.197     -1.441      0.150      -0.670       0.102
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity robust (HC1)
"""

In [ ]:
# Automated Creation of Indicator variables: The Category command
WatRegAll_07_1=tprstats.model("cs", "SUMMER_07~C(GROUP)", Water_Conservation_Data)
WatRegAll_07_1.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:              SUMMER_07   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                  0.000
No. Observations:              106669   F-statistic:                     14.99
Covariance Type:                  HC1   Prob (F-statistic):           9.43e-10
=================================================================================
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept        36.3528      0.282    129.105      0.000      35.801      36.905
C(GROUP)[T.2]    -0.9254      0.384     -2.413      0.016      -1.677      -0.174
C(GROUP)[T.3]    -1.4948      0.372     -4.014      0.000      -2.225      -0.765
C(GROUP)[T.4]     0.1197      0.302      0.396      0.692      -0.472       0.712
=================================================================================

Notes:
[1] Standard Errors are heteroscedasticity robust (HC1)
"""

The above command makes the lowest numbered group the reference category.

The `Treatment()` command lets us specify the reference category. Below, we make GROUP 4 the reference category

In [ ]:
WatRegAll_07_2=tprstats.model("cs", "SUMMER_07~C(GROUP, Treatment(4))", Water_Conservation_Data)
WatRegAll_07_2.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:              SUMMER_07   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                  0.000
No. Observations:              106669   F-statistic:                     14.99
Covariance Type:                  HC1   Prob (F-statistic):           9.43e-10
===============================================================================================
                                  coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
Intercept                      36.4725      0.109    333.728      0.000      36.258      36.687
C(GROUP, Treatment(4))[T.1]    -0.1197      0.302     -0.396      0.692      -0.712       0.472
C(GROUP, Treatment(4))[T.2]    -1.0451      0.282     -3.701      0.000      -1.599      -0.492
C(GROUP, Treatment(4))[T.3]    -1.6145      0.267     -6.045      0.000      -2.138      -1.091
===============================================================================================

Notes:
[1] Standard Errors are heteroscedasticity robust (HC1)
"""

The following regression investigates whether assignment to treatment was random. If yes, there should be
no significant difference in water consumption in the year prior to the experiment. We see that there was
not a significant difference, supporting the authors' claim of random assignment to treatment.

In [ ]:
WatReg06=tprstats.model("cs","WATER_2006~C(GROUP)",Water_Conservation_Data)
WatReg06.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:             WATER_2006   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                 -0.000
No. Observations:              106669   F-statistic:                    0.1129
Covariance Type:                  HC1   Prob (F-statistic):              0.953
=================================================================================
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept        58.4265      0.370    157.990      0.000      57.702      59.151
C(GROUP)[T.2]    -0.2509      0.531     -0.472      0.637      -1.293       0.791
C(GROUP)[T.3]     0.0091      0.528      0.017      0.986      -1.025       1.043
C(GROUP)[T.4]    -0.1283      0.401     -0.320      0.749      -0.914       0.657
=================================================================================

Notes:
[1] Standard Errors are heteroscedasticity robust (HC1)
"""

## Regression with Both Continuous and Indicator Variables

In [ ]:
# Import Parcel Delivery Data Set
Parcel_Delivery = pandas.read_excel("data/Parcel_Delivery.xlsx")

In [ ]:
# Parcel Delivery Regression
ParcelReg=tprstats.model("cs", "DELTIM~PARCELS+RB+RC", Parcel_Delivery)
ParcelReg.summary()

The following commands create the graph in the note set.

I include these plot commands for completeness, but I do not expect you to master these commands.

In [ ]:
Xnew = pandas.DataFrame({'PARCELS': [0,2], 'RB': [0,0], 'RC':[0,0] })

y_pred = ParcelReg.predict(Xnew)
plt.plot(Xnew['PARCELS'], y_pred, color = "red")
plt.xlim(0,2.5)
plt.ylim(15, 40)

def abline(intercept, slope):
    axes = plt.gca()
    x_vals = numpy.array(axes.get_xlim())
    y_vals = intercept + slope * x_vals
    plt.plot(x_vals, y_vals, '--')


abline(ParcelReg.params.iloc[0], ParcelReg.params.iloc[1])
abline((ParcelReg.params.iloc[0] + ParcelReg.params.iloc[2]), ParcelReg.params.iloc[1])
abline((ParcelReg.params.iloc[0] + ParcelReg.params.iloc[3]), ParcelReg.params.iloc[1])

plt.xlabel("Parcels")
plt.ylabel("Delivery Time")
plt.title("Delivery Time As a Function of Parcels for Three Routes")


In [ ]:
# Parcel Delivery Regression with factors based on ROUTE_NAME
ParcelRegFac= tprstats.model("cs", 'DELTIM~PARCELS+C(ROUTE_NAME)', Parcel_Delivery)
ParcelRegFac.summary()

In [ ]:
# Parcel Delivery with factors based on ROUTE_NUMBER
ParcelRegFac= tprstats.model("cs", 'DELTIM~PARCELS+C(ROUTE_NUMBER)', Parcel_Delivery)
ParcelRegFac.summary()

## Water Conservation Case: Analyze Summer 2009
Import Water_Conservation_Data.xlsx

In [ ]:
# Summer 2009 with only treatment variables
Wat2009Reg1=tprstats.model("cs", "SUMMER_09~TREAT1+TREAT2+TREAT3", Water_Conservation_Data)
Wat2009Reg1.summary()

In [ ]:
# Summer 2009 with treatment variables and variables determined prior to the experiment
Wat2009Reg2=tprstats.model("cs", "SUMMER_09~TREAT1+TREAT2+TREAT3+WATER_2006+APR_MAY_07", Water_Conservation_Data)
Wat2009Reg2.summary()

In [ ]:
#hypothesis = 'TREAT1=TREAT2'
WatRegAll_07.wald_test(("TREAT1=TREAT2"))

## Optional Appendices
The next two sets of commands show how to change the reference category using the `Treatment()` command.

In [ ]:
# Parcel Delivery with categories based on ROUTE_NAME and Route C the reference category
ParcelRegFac=tprstats.model("cs", "DELTIM ~ PARCELS + C(ROUTE_NAME,Treatment('RC'))", Parcel_Delivery)
ParcelRegFac.summary()

In the output from above regression, interpret the coefficient of `C(ROUTE_NAME, levels = Treatment("RC"))`. The graph in the slides on Indicator and Continuous variables will help.


In [ ]:
# Parcel Delivery with categories based on ROUTE_NUMBER and Route 2
# the reference category
ParcelRegFac2=tprstats.model("cs", "DELTIM ~ PARCELS + C(ROUTE_NUMBER, Treatment(2))", Parcel_Delivery)
ParcelRegFac2.summary()

In the output from above regression, interpret the coefficient of `C(ROUTE_NUMBER, Treatment(2))`. The graph in the slides on Indicator and Continuous variables will help.